# Predictive Analysis - Classification

The pipeline fits preprocessing within each cross-validation fold. Four classifiers use a stratified 72/18 split and five-fold tuning. Selection uses training CV accuracy, with a chronological check for drift.

Prepared with AI assistance. Data and outputs are reproducible.

In [1]:
"""Reproducible IBM Applied Data Science Capstone analysis.
Run: python analysis.py. Historical course snapshots are included in data/.
"""
from pathlib import Path
import json, sqlite3, io, contextlib, base64
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import folium
from folium.plugins import MarkerCluster, MousePosition
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, balanced_accuracy_score
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
DATA=ROOT/'data'; RESULTS=ROOT/'results'; RESULTS.mkdir(exist_ok=True)
plt.rcParams.update({'font.size':12,'axes.spines.top':False,'axes.spines.right':False})


def wrangle():
    df=pd.read_csv(DATA/'dataset_part_1.csv')
    bad={'False ASDS','False Ocean','False RTLS','None ASDS','None None'}
    df['Class']=(~df.Outcome.isin(bad)).astype(int)
    missing=df.isna().sum(); df['PayloadMass']=df.PayloadMass.fillna(df.PayloadMass.mean())
    assert len(df)==90 and df.FlightNumber.is_unique
    assert np.array_equal(df.Class,pd.read_csv(DATA/'dataset_part_2.csv').Class)
    df.to_csv(DATA/'analysis_dataset.csv',index=False)
    print('Published course API-derived checkpoint: 90 Falcon 9 launches')
    print('Date range:',df.Date.min(),'to',df.Date.max())
    print('Missing values before processing:\n',missing[missing>0].to_string())
    print('Outcome counts:\n',df.Outcome.value_counts().to_string())
    print('Class counts:',df.Class.value_counts().to_dict())
    return df


def saveplot(name):
    plt.tight_layout(); plt.savefig(RESULTS/(name+'.png'),dpi=170,bbox_inches='tight'); plt.close()


def prediction(df):
    # Outcomes and landing pad are excluded: they would reveal post-launch information.
    numeric=['FlightNumber','PayloadMass','Flights','Block','ReusedCount','GridFins','Reused','Legs']
    categorical=['Orbit','LaunchSite']
    pre=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),numeric),('cat',OneHotEncoder(handle_unknown='ignore',sparse_output=False),categorical)])
    X=df[numeric+categorical]; y=df.Class
    train,test=train_test_split(np.arange(len(df)),test_size=.2,random_state=2,stratify=y)
    cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
    configs={
      'Logistic regression':(LogisticRegression(max_iter=3000),{'model__C':[.01,.1,1,10]}),
      'SVM':(SVC(),{'model__C':[.1,1,10],'model__kernel':['linear','rbf'],'model__gamma':['scale','auto']}),
      'Decision tree':(DecisionTreeClassifier(random_state=42),{'model__max_depth':[2,3,4,6],'model__min_samples_leaf':[1,2,4]}),
      'KNN':(KNeighborsClassifier(),{'model__n_neighbors':[3,5,7,9],'model__weights':['uniform','distance']})}
    rows=[]; models={}
    for name,(model,grid) in configs.items():
        search=GridSearchCV(Pipeline([('pre',pre),('model',model)]),grid,cv=cv,scoring='accuracy',n_jobs=1)
        search.fit(X.iloc[train],y.iloc[train]); pred=search.predict(X.iloc[test]); models[name]=search
        rows.append({'Model':name,'CVAccuracy':search.best_score_,'TestAccuracy':accuracy_score(y.iloc[test],pred),'BalancedAccuracy':balanced_accuracy_score(y.iloc[test],pred),'Parameters':str(search.best_params_)})
    res=pd.DataFrame(rows); res.to_csv(RESULTS/'model_results.csv',index=False)
    best=res.sort_values('CVAccuracy',ascending=False).iloc[0].Model
    cm=confusion_matrix(y.iloc[test],models[best].predict(X.iloc[test]),labels=[0,1])
    fig,ax=plt.subplots(figsize=(8.8,4.4)); ix=np.arange(len(res)); ax.bar(ix-.18,res.CVAccuracy,.36,label='5-fold CV',color='#087f8c'); ax.bar(ix+.18,res.TestAccuracy,.36,label='Test',color='#e6a04b'); ax.set_xticks(ix,res.Model); ax.set_ylim(0,1.05); ax.set_ylabel('Accuracy'); ax.legend(); saveplot('model_accuracy')
    fig,ax=plt.subplots(figsize=(6,4.4)); ax.imshow(cm,cmap='Blues'); ax.set_xticks([0,1],['Failure / no recovery','Success']); ax.set_yticks([0,1],['Failure / no recovery','Success']); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    for i in range(2):
        for j in range(2): ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=26,color='white' if cm[i,j]>cm.max()/2 else 'black')
    saveplot('confusion_matrix')
    # Time split is a robustness check for changing launch technology.
    order=np.argsort(pd.to_datetime(df.Date)); old=order[:72]; recent=order[72:]
    temporal=models[best].best_estimator_; temporal.fit(X.iloc[old],y.iloc[old]); timeacc=accuracy_score(y.iloc[recent],temporal.predict(X.iloc[recent]))
    info={'best_model':best,'confusion_matrix':cm.tolist(),'test_size':len(test),'train_size':len(train),'baseline_test':float(y.iloc[test].mean()),'temporal_accuracy':timeacc,'temporal_baseline':float(y.iloc[recent].mean()),'test_indices':test.tolist()}
    print(res.to_string(index=False)); print(info); return info


In [2]:
df=wrangle()
model_info=prediction(df)

Published course API-derived checkpoint: 90 Falcon 9 launches
Date range: 2010-06-04 to 2020-11-05
Missing values before processing:
 LandingPad    26
Outcome counts:
 Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Class counts: {1: 60, 0: 30}
              Model  CVAccuracy  TestAccuracy  BalancedAccuracy                                                           Parameters
Logistic regression    0.861905      0.833333          0.791667                                                     {'model__C': 10}
                SVM    0.875238      0.722222          0.708333 {'model__C': 10, 'model__gamma': 'scale', 'model__kernel': 'linear'}
      Decision tree    0.848571      0.777778          0.791667                {'model__max_depth': 2, 'model__min_samples_leaf': 1}
                KNN    0.862857      0.777778          0.791667               {'model__n_neighbors': 3, 'model__weights